# Clean Audio Preprocessing

This notebook prepares a training-ready audio dataset from the raw `AudioWAV/` folder.

It is a replacement for the earlier preprocessing notebook and avoids the `processed_file_path` merge collision entirely.

Optional install:

```bash
pip install pandas numpy librosa soundfile tqdm
```

In [ ]:
from __future__ import annotations

import os
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import librosa
import numpy as np
import pandas as pd
import soundfile as sf
from tqdm.auto import tqdm


def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "AudioWAV").exists() and (candidate / "context.md").exists():
            return candidate
    raise FileNotFoundError("Could not find repo root containing AudioWAV/ and context.md")


ROOT = find_repo_root(Path.cwd().resolve())
RAW_MANIFEST = ROOT / "manifests" / "audio_manifest.csv"
OUTPUT_AUDIO_DIR = ROOT / "data" / "processed_audio"
OUTPUT_MANIFEST = ROOT / "manifests" / "processed" / "processed_audio_manifest.csv"
OUTPUT_AUDIO_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_MANIFEST.parent.mkdir(parents=True, exist_ok=True)

TARGET_SAMPLE_RATE = 16_000
TARGET_PEAK = 0.98
WRITE_SUBTYPE = "PCM_16"

EXCLUDED_DUPLICATE_FILE_NAMES = {
    "1006_TIE_HAP_XX.wav",
    "1006_TIE_NEU_XX.wav",
    "1013_WSI_DIS_XX.wav",
    "1013_WSI_SAD_XX.wav",
    "1017_IWW_ANG_XX.wav",
    "1017_IWW_FEA_XX.wav",
}

ROOT

In [ ]:
manifest = pd.read_csv(RAW_MANIFEST)
manifest["file_path"] = manifest["file_path"].map(lambda p: str(Path(p)))
manifest["is_excluded_duplicate"] = manifest["file_name"].isin(EXCLUDED_DUPLICATE_FILE_NAMES)

processed_manifest = manifest.loc[~manifest["is_excluded_duplicate"]].copy()
processed_manifest["source_file_path"] = processed_manifest["file_path"]
processed_manifest["processed_file_path"] = processed_manifest["clip_id"].map(
    lambda clip_id: str(OUTPUT_AUDIO_DIR / f"{clip_id}.wav")
)

print(f"Raw rows: {len(manifest):,}")
print(f"Excluded duplicate rows: {manifest['is_excluded_duplicate'].sum():,}")
print(f"Rows to process: {len(processed_manifest):,}")
processed_manifest[["split", "emotion"]].value_counts().sort_index().head(12)

## Standardize audio

This pipeline keeps preprocessing intentionally light:
- load as mono 16 kHz
- remove DC offset
- apply peak normalization
- write as 16-bit PCM WAV


In [ ]:
def standardize_audio(source_path: str, target_path: str, target_sr: int = TARGET_SAMPLE_RATE) -> dict:
    source = Path(source_path)
    target = Path(target_path)
    target.parent.mkdir(parents=True, exist_ok=True)

    audio, _ = librosa.load(source, sr=target_sr, mono=True)
    audio = audio.astype(np.float32, copy=False)
    audio = audio - float(np.mean(audio))

    peak = float(np.max(np.abs(audio))) if audio.size else 0.0
    if peak > 0:
        audio = (audio / peak) * TARGET_PEAK

    sf.write(target, audio, target_sr, subtype=WRITE_SUBTYPE)
    return {
        "processed_sample_rate": target_sr,
        "processed_num_samples": len(audio),
        "processed_duration_sec": round(len(audio) / float(target_sr), 6),
        "status": "ok",
    }


def process_row(row: dict) -> dict:
    try:
        return {
            "clip_id": row["clip_id"],
            **standardize_audio(row["source_file_path"], row["processed_file_path"]),
        }
    except Exception as exc:  # noqa: BLE001
        return {
            "clip_id": row["clip_id"],
            "processed_sample_rate": np.nan,
            "processed_num_samples": np.nan,
            "processed_duration_sec": np.nan,
            "status": f"error: {exc}",
        }


records = processed_manifest.to_dict(orient="records")
results = []
max_workers = min(8, os.cpu_count() or 4)

with ThreadPoolExecutor(max_workers=max_workers) as executor:
    for result in tqdm(executor.map(process_row, records), total=len(records)):
        results.append(result)

results_df = pd.DataFrame(results)
results_df["status"].value_counts()

In [ ]:
results_df = results_df.set_index("clip_id")
processed_manifest = processed_manifest.join(results_df, on="clip_id", rsuffix="_result")

required_columns = {"processed_file_path", "processed_duration_sec", "status"}
missing_columns = required_columns - set(processed_manifest.columns)
if missing_columns:
    raise KeyError(f"Missing expected columns: {sorted(missing_columns)}")

processed_manifest = processed_manifest.loc[processed_manifest["status"] == "ok"].copy()
processed_manifest.to_csv(OUTPUT_MANIFEST, index=False)

summary = {
    "processed_rows": len(processed_manifest),
    "excluded_duplicate_rows": int(manifest["is_excluded_duplicate"].sum()),
    "failed_rows": int((results_df["status"] != "ok").sum()),
    "output_manifest": str(OUTPUT_MANIFEST),
    "output_audio_dir": str(OUTPUT_AUDIO_DIR),
}
summary

In [ ]:
display_columns = [
    "clip_id",
    "emotion",
    "split",
    "source_file_path",
    "processed_file_path",
    "processed_duration_sec",
]
processed_manifest[display_columns].head()